# Walk-Forward: TA + ML + LLM

**Идея:**
1. Обучаем ML модели
2. Получаем предсказания от TA и ML
3. Отправляем результаты в LLM с метриками
4. LLM даёт своё мнение — согласен или нет
5. Если TA + ML + LLM согласны → учитываем

Консенсус 3х источников!

## Шаг 1: Подготовка

In [24]:
import os
import sys
import requests
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

from streamlit_app.grading import calculate_signal_grade

load_dotenv()

LLM_URL = "http://192.168.1.7:11434/api/chat"
LLM_MODEL = "llama3"
LLM_TIMEOUT = 30

DB_HOST = "127.0.0.1"
DB_PORT = 5432
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PASSWORD = "postgres"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

print(f"LLM: {LLM_MODEL}")
print(f"BD: {DB_HOST}:{DB_PORT}/{DB_NAME}")

LLM: llama3
BD: 127.0.0.1:5432/postgres


## Шаг 2: ML модели

In [25]:
from models.ridge import RidgeTradeModel
from models.xgboost_model import XGBoostTradeModelNew
from models.lightgbm_model import LightGBMTradeModelNew
from models.cat_boost_model import CatBoostTradeModelNew
from models.random_forest_regression_model import RandomForestTradeModelNew
from models.rf_classifier import RandomForestClassifierNew

ML_MODELS = {
    'ridge': RidgeTradeModel,
    'xgboost': XGBoostTradeModelNew,
    'lightgbm': LightGBMTradeModelNew,
    'catboost': CatBoostTradeModelNew,
    'random_forest': RandomForestTradeModelNew,
    'rf_classifier': RandomForestClassifierNew,
}

print(f"ML: {list(ML_MODELS.keys())}")

ML: ['ridge', 'xgboost', 'lightgbm', 'catboost', 'random_forest', 'rf_classifier']


## Шаг 3: Загрузка данных

In [26]:
TEST_TICKERS = ["SBER", "OZON", "VTBR"]

tickers_str = "', '".join(TEST_TICKERS)
query = text(f"SELECT ticker, figi, name FROM public.tickers WHERE ticker IN ('{tickers_str}')")

with engine.connect() as conn:
    tickers_df = pd.read_sql(query, conn)

ticker_to_figi = dict(zip(tickers_df['ticker'], tickers_df['figi']))
print("Tickers:", ticker_to_figi)

def load_candles_from_db(figi):
    q = text(f'SELECT timestamp, open, high, low, close, volume FROM all_dfs."{figi}" ORDER BY timestamp')
    df = pd.read_sql(q, engine)
    if not df.empty:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

candles_data = {}
for ticker, figi in ticker_to_figi.items():
    df = load_candles_from_db(figi)
    if not df.empty:
        candles_data[ticker] = df
        print(f"{ticker}: {len(df)} candles")

Tickers: {'SBER': 'BBG004730N88', 'OZON': 'TCS80A10CW95', 'VTBR': 'BBG004730ZJ9'}
SBER: 787 candles
OZON: 696 candles
VTBR: 783 candles


## Шаг 4: Индикаторы TA

In [27]:
def calculate_indicators(df):
    df = df.copy()
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = np.where(loss != 0, gain / loss, 100)
    df['rsi'] = 100 - (100 / (1 + rs))
    
    ema_12 = df['close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema_12 - ema_26
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    
    df['sma_5'] = df['close'].rolling(window=5).mean()
    df['sma_20'] = df['close'].rolling(window=20).mean()
    
    df['bb_middle'] = df['close'].rolling(window=20).mean()
    bb_std = df['close'].rolling(window=20).std()
    df['bb_upper'] = df['bb_middle'] + (bb_std * 2)
    df['bb_lower'] = df['bb_middle'] - (bb_std * 2)
    
    low_min = df['low'].rolling(window=14).min()
    high_max = df['high'].rolling(window=14).max()
    df['stoch_k'] = 100 * ((df['close'] - low_min) / (high_max - low_min))
    
    df['price_vs_sma20'] = df['close'] / df['sma_20'] - 1
    df['next_return'] = df['close'].shift(-1) / df['close'] - 1
    
    return df

def generate_ta_signals(df):
    df = df.copy()
    df['signal_rsi'] = 0
    df.loc[df['rsi'] < 30, 'signal_rsi'] = 1
    df.loc[df['rsi'] > 70, 'signal_rsi'] = -1
    
    df['macd_prev'] = df['macd'].shift(1)
    df['macd_signal_prev'] = df['macd_signal'].shift(1)
    df['signal_macd'] = 0
    df.loc[(df['macd'] > df['macd_signal']) & (df['macd_prev'] <= df['macd_signal_prev']), 'signal_macd'] = 1
    df.loc[(df['macd'] < df['macd_signal']) & (df['macd_prev'] >= df['macd_signal_prev']), 'signal_macd'] = -1
    
    df['sma_5_prev'] = df['sma_5'].shift(1)
    df['sma_20_prev'] = df['sma_20'].shift(1)
    df['signal_sma_cross'] = 0
    df.loc[(df['sma_5'] > df['sma_20']) & (df['sma_5_prev'] <= df['sma_20_prev']), 'signal_sma_cross'] = 1
    df.loc[(df['sma_5'] < df['sma_20']) & (df['sma_5_prev'] >= df['sma_20_prev']), 'signal_sma_cross'] = -1
    
    df['signal_trend'] = 0
    df.loc[df['price_vs_sma20'] > 0.02, 'signal_trend'] = 1
    df.loc[df['price_vs_sma20'] < -0.02, 'signal_trend'] = -1
    
    df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
    df['signal_bb'] = 0
    df.loc[df['bb_position'] < 0.1, 'signal_bb'] = 1
    df.loc[df['bb_position'] > 0.9, 'signal_bb'] = -1
    
    df['signal_stoch'] = 0
    df.loc[df['stoch_k'] < 20, 'signal_stoch'] = 1
    df.loc[df['stoch_k'] > 80, 'signal_stoch'] = -1
    
    return df

for ticker in candles_data:
    candles_data[ticker] = calculate_indicators(candles_data[ticker])
    candles_data[ticker] = generate_ta_signals(candles_data[ticker])

print("TA indicators ready")

TA indicators ready


## Шаг 5: Функции ML и LLM

In [28]:
LLM_SYSTEM_PROMPT = """Ты — опытный финансовый аналитик с 20-летним стажем торговли на фондовом рынке.

ЗАДАЧА:
Ты получаешь данные от ML моделей, сигнал от технического анализа и новости о компании за день сигнала. Оцени надёжность предсказания.

ДАННЫЕ:
1. Название акции (тикер)
2. Текущая цена
3. Предсказания ML моделей (название, сигнал BUY/SELL, R2, Direction Accuracy)
4. Консенсус ML моделей
5. Сигнал технического анализа (какой индикатор)
6. Значения индикаторов (RSI, MACD, Stochastic)
7. Новости о компании за день сигнала

МЕТРИКИ:
- R2 > 0.7 отлично, 0.5-0.7 хорошо, <0.5 слабо
- Direction > 60% отлично, 55-60% хорошо, <55% слабо

ПРАВИЛА:
- AGREE: ML согласны + хорошие метрики + TA подтверждает + новости нейтральные или позитивные для сигнала
- DISAGREE: ML не согласны / низкие метрики / TA противоречит / негативные новости противоречат сигналу

ОТВЕТ: Только AGREE или DISAGREE. Без текста."""


import feedparser
from datetime import datetime, timedelta
from bs4 import BeautifulSoup

# Кэш новостей (для последних новостей - не используется)
NEWS_CACHE = {}


def get_company_news_by_date(company, target_date, max_pages=50):
    """
    Получает новости компании за конкретную дату с smart-lab.ru.
    
    Args:
        company: 'sber', 'ozon', 'vtb'
        target_date: datetime.date или 'YYYY-MM-DD'
        max_pages: максимум страниц для парсинга
    
    Returns:
        list: список новостей [{'date': str, 'title': str}, ...]
    """
    # Парсим дату
    if isinstance(target_date, str):
        target_date = datetime.strptime(target_date, '%Y-%m-%d').date()
    
    company_search = {'sber': 'сбер', 'ozon': 'ozon', 'vtb': 'втб'}
    search = company_search.get(company.lower(), company.lower())
    
    headers = {'User-Agent': 'Mozilla/5.0'}
    all_news = []
    
    month_map = {'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
                'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
                'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12}
    
    for page in range(1, max_pages + 1):
        url = f'https://smart-lab.ru/search/topics/page{page}/?q={search}&blog=news'
        if page == 1:
            url = f'https://smart-lab.ru/search/topics/?blog=news&q={search}'
        
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            soup = BeautifulSoup(resp.content, 'lxml')
            
            topics = soup.find_all('div', class_=lambda x: x and 'topic' in x if x else False)
            if not topics:
                break
            
            for t in topics:
                title_elem = t.find('h2', class_='title')
                if title_elem:
                    link = title_elem.find('a')
                    if link:
                        title = link.get('title', '') or link.get_text(strip=True)
                        date_elem = t.find('li', class_='date')
                        date_str = date_elem.get_text(strip=True)[:20] if date_elem else ''
                        
                        # Парсим дату
                        pub_date = None
                        try:
                            parts = date_str.replace(',', '').split()
                            if len(parts) >= 3:
                                day = int(parts[0])
                                month = month_map.get(parts[1], 1)
                                year = int(parts[2])
                                pub_date = datetime(year, month, day).date()
                        except:
                            pass
                        
                        # Фильтр по дате
                        if pub_date != target_date:
                            continue
                        
                        all_news.append({'date': date_str, 'title': title})
            
            if len(all_news) >= 3:
                break
                
        except:
            break
    
    return all_news


def query_llm(ticker, current_price, ml_results, ta_signal, ta_indicator, indicators_values, signal_date=None, news_list=None):
    """
    Запрашивает LLM с новостями за день сигнала.
    
    Args:
        ticker: тикер (SBER, OZON, VTBR)
        current_price: текущая цена
        ml_results: результаты ML моделей
        ta_signal: сигнал TA (1, -1, 0)
        ta_indicator: название индикатора
        indicators_values: значения индикаторов
        signal_date: дата сигнала (datetime.date) - для получения новостей за этот день
        news_list: готовый список новостей (если передан - не запрашиваем заново)
    """
    # Формируем информацию о ML
    models_info = []
    for name, data in ml_results.items():
        s = {1: "BUY", -1: "SELL", 0: "HOLD"}[data['signal']]
        models_info.append(f"{name}: {s}, R2={data.get('r2', 0):.2f}, Dir={data.get('direction_accuracy', 50):.0f}%")
    ml_summary = ", ".join(models_info)
    
    # Консенсус ML
    signals = [d['signal'] for d in ml_results.values()]
    buy_votes = signals.count(1)
    sell_votes = signals.count(-1)
    ml_consensus = "BUY" if buy_votes > sell_votes else "SELL" if sell_votes > buy_votes else "HOLD"
    
    # TA сигнал
    ta_s = {1: "BUY", -1: "SELL", 0: "NEUTRAL"}[ta_signal]
    
    # Значения индикаторов
    rsi = indicators_values.get('rsi', 0) or 0
    macd = indicators_values.get('macd', 0) or 0
    stoch = indicators_values.get('stoch_k', 0) or 0
    
    # Форматируем дату
    if signal_date:
        if hasattr(signal_date, 'strftime'):
            signal_date_str = signal_date.strftime('%Y-%m-%d')
        else:
            signal_date_str = str(signal_date)
    else:
        signal_date_str = "?"
    
    # Используем переданный список новостей или запрашиваем новые
    if news_list is None:
        company_map = {'SBER': 'sber', 'OZON': 'ozon', 'VTBR': 'vtb'}
        company = company_map.get(ticker, ticker.lower())
        news_list = get_company_news_by_date(company, signal_date, max_pages=30)
    
    # Форматируем новости
    if news_list:
        news_str = "\n".join([f"- {n['title'][:120]}" for n in news_list[:5]])
    else:
        news_str = "Новостей за этот день не найдено"
    
    user_msg = f"""Акция: {ticker}, Цена: {current_price:.2f}
ML ({ml_consensus}): {ml_summary}
TA: {ta_s} ({ta_indicator}). RSI={rsi:.0f}, MACD={macd:.2f}, Stoch={stoch:.0f}
Новости за день сигнала ({signal_date_str}):
{news_str}

AGREE или DISAGREE?"""
    
    try:
        resp = requests.post(LLM_URL, json={
            "model": LLM_MODEL,
            "messages": [
                {"role": "system", "content": LLM_SYSTEM_PROMPT},
                {"role": "user", "content": user_msg}
            ],
            "stream": False
        }, timeout=LLM_TIMEOUT)
        
        if resp.status_code == 200:
            answer = resp.json()['message']['content'].strip().upper()
            if "AGREE" in answer:
                return 1
            elif "DISAGREE" in answer:
                return -1
    except Exception as e:
        print(f"LLM error: {e}")
    
    return 0


def train_ml_models(train_df, test_df):
    results = {}
    if len(train_df) < 200:
        return results
    
    for name, ModelClass in ML_MODELS.items():
        try:
            model = ModelClass(test_size=0.2, random_state=42)
            model.train(train_df)
            pred = model.predict_next(test_df)
            s_map = {'BUY': 1, 'SELL': -1, 'HOLD': 0, 'NEUTRAL': 0}
            m = model.test_metrics or {}
            results[name] = {
                'signal': s_map.get(pred['signal'], 0),
                'r2': m.get('test_r2', 0) or 0,
                'direction_accuracy': m.get('test_direction_accuracy', 50) or 50,
            }
        except:
            pass
    
    return results


def get_ml_consensus(ml_results, min_ml=2):
    if len(ml_results) < min_ml:
        return {'signal': 0, 'consensus_pct': 0}
    
    signals = [r['signal'] for r in ml_results.values()]
    total = len(signals)
    buy = signals.count(1)
    sell = signals.count(-1)
    
    if buy > total / 2:
        return {'signal': 1, 'consensus_pct': buy / total * 100}
    elif sell > total / 2:
        return {'signal': -1, 'consensus_pct': sell / total * 100}
    return {'signal': 0, 'consensus_pct': max(buy, sell) / total * 100}


print("Functions ready: query_llm (+ news by date), train_ml_models, get_ml_consensus")

Functions ready: query_llm (+ news by date), train_ml_models, get_ml_consensus


## Шаг 6: Walk-Forward

In [29]:
TRAIN_WINDOW = 300
STEP = 1
USE_LLM = False
USE_DL = True  # Включить DL модели!
PROJECT_ROOT = os.path.abspath('.')

# print(f"[DEBUG] PROJECT_ROOT = {PROJECT_ROOT}")

import subprocess
import re

def get_dl_predictions(figi: str, train_df: pd.DataFrame) -> dict:
    """
    Запускает DL модели (LSTM, TCN) через subprocess
    """
    if not USE_DL:
        return {}
    
    results = {}
    
    for model_name in ['lstm', 'tcn']:
        try:
            # print(f"[DEBUG] Запуск {model_name} для {figi}...")
            
            script = f'''
import sys
sys.path.insert(0, "{PROJECT_ROOT}")
from models.{model_name}_model import main
import os
os.environ["DB_HOST"] = "127.0.0.1"
os.environ["DB_PORT"] = "5432"
os.environ["DB_NAME"] = "postgres"
os.environ["DB_USER"] = "postgres"
os.environ["DB_PASSWORD"] = "postgres"
main("{figi}")
'''
            result = subprocess.run(
                ['python', '-c', script],
                capture_output=True,
                text=True,
                timeout=180,
                cwd=PROJECT_ROOT
            )
            
            output = result.stdout + result.stderr
            
            # Отладочный вывод
            # if result.returncode != 0:
            #     print(f"  [ERROR] {model_name} returncode={result.returncode}")
            #     print(f"  [STDERR] {result.stderr[:500]}")
            
            signal_match = re.search(r'Торговый сигнал:\s*(\w+)', output)
            r2_match = re.search(r'R²:\s*([-\d.]+)', output)
            dir_match = re.search(r'Direction Accuracy:\s*([\d.]+)', output)
            
            signal_map = {'BUY': 1, 'SELL': -1, 'HOLD': 0, 'NEUTRAL': 0}
            
            if signal_match:
                results[model_name] = {
                    'signal': signal_map.get(signal_match.group(1), 0),
                    'r2': float(r2_match.group(1)) if r2_match else 0,
                    'direction_accuracy': float(dir_match.group(1)) if dir_match else 50,
                }
                # print(f"  [OK] {model_name}: signal={signal_match.group(1)}, r2={results[model_name]['r2']:.3f}")
            else:
                pass
                # print(f"  [WARN] {model_name}: сигнал не найден в выводе")
                # print(f"  [OUTPUT] {output[-300:]}")
        except Exception as e:
            print(f"DL {model_name} error: {e}")
            continue
    
    return results


signal_columns = ['signal_rsi', 'signal_macd', 'signal_sma_cross', 'signal_trend', 'signal_bb', 'signal_stoch']
indicator_names = {
    'signal_rsi': 'RSI',
    'signal_macd': 'MACD',
    'signal_sma_cross': 'SMA',
    'signal_trend': 'Trend',
    'signal_bb': 'BB',
    'signal_stoch': 'Stoch'
}

all_results = {}

# Для расчета Sharpe Ratio - собираем доходности
ta_only_returns = []  # Доходности только по TA сигналам
ta_ml_llm_returns = []  # Доходности по TA+ML+LLM сигналам

# Кэш новостей по датам (чтобы не запрашивать много раз)
news_by_date_cache = {}

for ticker, df in candles_data.items():
    print(f"\n=== {ticker} ===")
    
    ticker_results = {sig_col: {'total': 0, 'correct': 0, 'accuracy': 0, 'llm_agree': 0, 'returns': []} for sig_col in signal_columns}
    
    llm_cache = {}
    dl_cache = {}
    
    for i in range(TRAIN_WINDOW, len(df) - 1, STEP):
        train_df = df.iloc[:i].copy()
        test_df = df.iloc[:i+1].copy()
        
        # ML модели
        ml_results = train_ml_models(train_df, test_df)
        
        # DL модели
        if USE_DL:
            figi = ticker_to_figi[ticker]
            dl_key = (ticker, i)
            if dl_key not in dl_cache:
                dl_results = get_dl_predictions(figi, train_df)
                dl_cache[dl_key] = dl_results
            else:
                dl_results = dl_cache[dl_key]
        else:
            dl_results = {}
        
        # Консенсус ML + DL
        all_model_results = {**ml_results, **dl_results}
        ml_consensus = get_ml_consensus(all_model_results)
        
        if ml_consensus['signal'] == 0:
            continue
        
        test_row = df.iloc[i]
        current_price = test_row['close']
        next_return = test_row['next_return']
        
        if pd.isna(next_return):
            continue
        
        # Дата сигнала (для получения новостей за этот день)
        signal_date = test_row['timestamp'].date() if 'timestamp' in test_row else None
        
        for sig_col in signal_columns:
            ta_signal = test_row[sig_col]
            
            if ta_signal == 0:
                continue
            
            # TA only: все сигналы TA
            ta_only_returns.append({
                'ticker': ticker,
                'signal': ta_signal,
                'return': next_return * ta_signal  # Положительный если сигнал правильный
            })
            
            # TA + ML: только если ML консенсус совпадает с TA
            if ml_consensus['signal'] != ta_signal:
                continue
            
            # LLM проверка - ТОЛЬКО если есть новости за этот день
            if USE_LLM:
                if signal_date is None:
                    continue
                
                cache_key = (ticker, i, sig_col)
                
                if cache_key not in llm_cache:
                    # Сначала проверяем есть ли новости за этот день
                    company_map = {'SBER': 'sber', 'OZON': 'ozon', 'VTBR': 'vtb'}
                    company = company_map.get(ticker, ticker.lower())
                    
                    # Проверяем кэш новостей
                    news_cache_key = (company, signal_date)
                    if news_cache_key not in news_by_date_cache:
                        news_list = get_company_news_by_date(company, signal_date, max_pages=50)
                        news_by_date_cache[news_cache_key] = news_list
                    else:
                        news_list = news_by_date_cache[news_cache_key]
                    
                    # Если новостей нет - пропускаем этот сигнал!
                    if not news_list:
                        llm_cache[cache_key] = None  # Нет новостей - не проверяем LLM
                        continue
                    
                    # Есть новости - запрашиваем LLM с метриками
                    ind_vals = {'rsi': test_row.get('rsi'), 'macd': test_row.get('macd'), 'stoch_k': test_row.get('stoch_k')}
                    llm_resp = query_llm(ticker, current_price, ml_results, ta_signal, indicator_names[sig_col], ind_vals, signal_date=signal_date, news_list=news_list)
                    llm_cache[cache_key] = llm_resp
                else:
                    llm_resp = llm_cache[cache_key]
                
                # Если LLM не согласен - пропускаем
                if llm_resp != ta_signal:
                    continue
                
                ticker_results[sig_col]['llm_agree'] += 1
            
            # TA + ML + LLM: только если LLM согласен
            ticker_results[sig_col]['total'] += 1
            ta_ml_llm_returns.append({
                'ticker': ticker,
                'signal': ta_signal,
                'return': next_return * ta_signal
            })
            
            if (ta_signal == 1 and next_return > 0) or (ta_signal == -1 and next_return < 0):
                ticker_results[sig_col]['correct'] += 1
    
    for sig_col in ticker_results:
        if ticker_results[sig_col]['total'] > 0:
            ticker_results[sig_col]['accuracy'] = ticker_results[sig_col]['correct'] / ticker_results[sig_col]['total'] * 100
    
    all_results[ticker] = ticker_results
    
    print(f"{'Ind':<10} {'Agree':>8} {'Correct':>8} {'Acc%':>8} {'LLM OK':>8}")
    for sig_col in signal_columns:
        r = ticker_results[sig_col]
        print(f"{indicator_names[sig_col]:<10} {r['total']:>8} {r['correct']:>8} {r['accuracy']:>7.1f} {r['llm_agree']:>8}")

print(f"\nDone! LLM={USE_LLM}, DL={USE_DL}")
print(f"TA only signals: {len(ta_only_returns)}")
print(f"TA+ML+LLM signals: {len(ta_ml_llm_returns)}")
print(f"News cache size: {len(news_by_date_cache)}")


=== SBER ===
Ind           Agree  Correct     Acc%   LLM OK
RSI              48       37    77.1        0
MACD             10        9    90.0        0
SMA               8        7    87.5        0
Trend            33       31    93.9        0
BB               38       29    76.3        0
Stoch            81       64    79.0        0

=== OZON ===
Ind           Agree  Correct     Acc%   LLM OK
RSI              65       44    67.7        0
MACD             10        6    60.0        0
SMA               5        3    60.0        0
Trend            80       58    72.5        0
BB               55       39    70.9        0
Stoch           103       66    64.1        0

=== VTBR ===
Ind           Agree  Correct     Acc%   LLM OK
RSI              85       52    61.2        0
MACD              9        5    55.6        0
SMA              17       12    70.6        0
Trend           100       76    76.0        0
BB               60       39    65.0        0
Stoch            89       53    59.

## Шаг 7: Сравнение + Sharpe Ratio

In [30]:
def evaluate_ta_only(df, sig_col):
    sdf = df[(df[sig_col] != 0) & (df['next_return'].notna())].copy()
    if sdf.empty:
        return {'total': 0, 'accuracy': 0}
    buy = sdf[sdf[sig_col] == 1]
    cb = (buy['next_return'] > 0).sum() if len(buy) > 0 else 0
    sell = sdf[sdf[sig_col] == -1]
    cs = (sell['next_return'] < 0).sum() if len(sell) > 0 else 0
    total = len(buy) + len(sell)
    return {'total': total, 'accuracy': (cb + cs) / total * 100 if total > 0 else 0}


def calculate_sharpe_ratio(returns: list, risk_free_rate: float = 0.0) -> float:
    """
    Рассчитывает коэффициент Шарпа.
    
    Args:
        returns: список доходностей (в долях, не в процентах)
        risk_free_rate: безрисковая ставка (годовая)
    
    Returns:
        Sharpe Ratio (годовой, если данные дневные)
    """
    if len(returns) < 2:
        return 0.0
    
    returns_arr = np.array(returns)
    
    # Средняя доходность (дневная)
    mean_return = np.mean(returns_arr)
    
    # Стандартное отклонение (дневное)
    std_return = np.std(returns_arr, ddof=1)
    
    if std_return == 0:
        return 0.0
    
    # Годовой Sharpe (252 торговых дня)
    sharpe = (mean_return - risk_free_rate / 252) / std_return * np.sqrt(252)
    
    return sharpe


def calculate_metrics(returns: list, label: str = "") -> dict:
    """Рассчитывает основные метрики для списка доходностей."""
    if not returns:
        return {'total': 0, 'mean': 0, 'std': 0, 'sharpe': 0, 'win_rate': 0, 'total_return': 0}
    
    returns_arr = np.array(returns)
    
    wins = (returns_arr > 0).sum()
    total = len(returns_arr)
    
    return {
        'total': total,
        'mean': np.mean(returns_arr) * 100,  # в процентах
        'std': np.std(returns_arr, ddof=1) * 100,  # в процентах
        'sharpe': calculate_sharpe_ratio(returns_arr),
        'win_rate': wins / total * 100 if total > 0 else 0,
        'total_return': (np.prod(1 + returns_arr) - 1) * 100 if total > 0 else 0  # кумулятивная доходность
    }


# Рассчитываем метрики для TA only и TA+ML+LLM
ta_returns = [r['return'] for r in ta_only_returns]
tmll_returns = [r['return'] for r in ta_ml_llm_returns]

ta_metrics = calculate_metrics(ta_returns, "TA Only")
tmll_metrics = calculate_metrics(tmll_returns, "TA+ML+LLM")

print("=" * 70)
print("МЕТРИКИ: TA Only vs TA + ML + LLM")
print("=" * 70)

print(f"\n{'Метрика':<20} {'TA Only':>15} {'TA+ML+LLM':>15}")
print("-" * 50)
print(f"{'Сигналов':<20} {ta_metrics['total']:>15} {tmll_metrics['total']:>15}")
print(f"{'Win Rate %':<20} {ta_metrics['win_rate']:>14.1f}% {tmll_metrics['win_rate']:>14.1f}%")
print(f"{'Средняя доходность %':<20} {ta_metrics['mean']:>14.2f}% {tmll_metrics['mean']:>14.2f}%")
print(f"{'Std отклонение %':<20} {ta_metrics['std']:>14.2f}% {tmll_metrics['std']:>14.2f}%")
print(f"{'Sharpe Ratio':<20} {ta_metrics['sharpe']:>15.2f} {tmll_metrics['sharpe']:>15.2f}")
print(f"{'Кумулятивная доходность %':<20} {ta_metrics['total_return']:>14.2f}% {tmll_metrics['total_return']:>14.2f}%")

print("\n" + "=" * 70)
print("Сравнение: TA vs TA + ML + LLM")
print("=" * 70)

comparison = []

for ticker, df in candles_data.items():
    print(f"\n{ticker}:")
    for sig_col in signal_columns:
        ta = evaluate_ta_only(df, sig_col)
        tmll = all_results[ticker].get(sig_col, {'total': 0, 'accuracy': 0})
        comparison.append({
            'ticker': ticker,
            'indicator': indicator_names[sig_col],
            'ta_accuracy': ta['accuracy'],
            'ta_ml_llm_signals': tmll['total'],
            'ta_ml_llm_accuracy': tmll['accuracy'],
        })

comp_df = pd.DataFrame(comparison)

summary = comp_df.groupby('indicator').agg({
    'ta_accuracy': 'mean',
    'ta_ml_llm_signals': 'sum',
    'ta_ml_llm_accuracy': 'mean',
}).round(1)

print(summary.to_string())

avg_ta = comp_df['ta_accuracy'].mean()
avg_tmll = comp_df['ta_ml_llm_accuracy'].mean()

print(f"\nTA avg accuracy: {avg_ta:.1f}%")
print(f"TA+ML+LLM avg accuracy: {avg_tmll:.1f}%")
print(f"Improvement: {avg_tmll - avg_ta:+.1f}%")

МЕТРИКИ: TA Only vs TA + ML + LLM

Метрика                      TA Only       TA+ML+LLM
--------------------------------------------------
Сигналов                        1701             896
Win Rate %                     50.6%           70.3%
Средняя доходность %           0.05%           0.90%
Std отклонение %               2.54%           2.35%
Sharpe Ratio                    0.31            6.07
Кумулятивная доходность %          32.88%      239602.61%

Сравнение: TA vs TA + ML + LLM

SBER:

OZON:

VTBR:
           ta_accuracy  ta_ml_llm_signals  ta_ml_llm_accuracy
indicator                                                    
BB                52.7                153                70.7
MACD              49.9                 29                68.5
RSI               51.8                198                68.7
SMA               50.5                 30                72.7
Stoch             50.1                273                67.5
Trend             50.8                213          

In [31]:
import requests
from bs4 import BeautifulSoup

def get_company_news_smartlab(ticker: str, max_news: int = 3) -> str:
    """
    Получает последние новости о компании со smart-lab.ru через HTML парсинг.
    """
    # Маппинг тикеров на названия для поиска
    ticker_to_search = {
        'SBER': 'сбер',
        'OZON': 'ozon',
        'VTBR': 'втб',
    }
    
    search_term = ticker_to_search.get(ticker, ticker.lower())
    url = f"https://smart-lab.ru/search/topics/?blog=news&q={search_term}"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, "lxml")
        news_items = []
        
        # Ищем блоки новостей - класс topic bluid_
        topics = soup.find_all("div", class_=lambda x: x and "topic" in x)
        
        for topic in topics[:max_news]:
            # Заголовок новости
            title_elem = topic.find("h2", class_="title")
            if title_elem and title_elem.find("a"):
                title = title_elem.find("a").get_text(strip=True)[:100]
            
            # Дата
            date_elem = topic.find("li", class_="date")
            if date_elem:
                date = date_elem.get_text(strip=True)[:16]
            else:
                date = ""
            
            if title:
                news_items.append(f"- {date}: {title}")
        
        result = "\n".join(news_items) if news_items else "Новостей нет"
        
    except Exception as e:
        result = f"Ошибка: {e}"
    
    return result


# Тест
for ticker in ['SBER', 'OZON', 'VTBR']:
    news = get_company_news_smartlab(ticker, max_news=3)
    print(f"\n=== {ticker} ===")
    print(news)


=== SBER ===
- 08 мая 2026, 14:: Западное правило «Sell in May and go away» не работает в России — директор по маркетингу УК Гамма Гр
- 07 мая 2026, 17:: Инвестбанк "Синара" заменил акции Сбербанка на Роснефть и ДОМ.РФ в своём модельном портфеле, зафикси
- 05 мая 2026, 12:: Сейчас денег нет ни у кого. Зато есть различные финансовые инструменты — и мы будем их использовать:

=== OZON ===
- 07 мая 2026, 17:: Инвестбанк "Синара" заменил акции Сбербанка на Роснефть и ДОМ.РФ в своём модельном портфеле, зафикси
- 05 мая 2026, 12:: Сейчас денег нет ни у кого. Зато есть различные финансовые инструменты — и мы будем их использовать:
- 05 мая 2026, 11:: Аналитики ПСБ представили подборку лучших акций для покупки в мае 2026 года

=== VTBR ===
- 08 мая 2026, 14:: Западное правило «Sell in May and go away» не работает в России — директор по маркетингу УК Гамма Гр
- 08 мая 2026, 10:: Анатолий Аксаков планирует обратиться в Минцифры с предложением включить все банки в "белые списки" 
- 08 мая 2026, 

In [ ]:
# Функция получения новостей за конкретную дату с smart-lab.ru
import requests
from bs4 import BeautifulSoup
from datetime import datetime

NEWS_CACHE_DATE = {}


def get_company_news_by_date(company, target_date, max_pages=200):
    """
    Получает новости компании за конкретную дату.
    
    Args:
        company: 'sber', 'ozon', 'vtb'
        target_date: datetime.date или 'YYYY-MM-DD'
        max_pages: максимум страниц для парсинга (200 страниц ~ 2000 новостей)
    
    Returns:
        list: список новостей [{'date': str, 'title': str}, ...]
    """
    # Парсим дату
    if isinstance(target_date, str):
        target_date = datetime.strptime(target_date, '%Y-%m-%d').date()
    
    cache_key = (company, str(target_date), max_pages)
    if cache_key in NEWS_CACHE_DATE:
        return NEWS_CACHE_DATE[cache_key]
    
    company_search = {'sber': 'сбер', 'ozon': 'ozon', 'vtb': 'втб'}
    search = company_search.get(company.lower(), company.lower())
    
    headers = {'User-Agent': 'Mozilla/5.0'}
    all_news = []
    
    month_map = {'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
                'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
                'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12}
    
    for page in range(1, max_pages + 1):
        url = f'https://smart-lab.ru/search/topics/page{page}/?q={search}&blog=news'
        if page == 1:
            url = f'https://smart-lab.ru/search/topics/?blog=news&q={search}'
        
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            soup = BeautifulSoup(resp.content, 'lxml')
            
            topics = soup.find_all('div', class_=lambda x: x and 'topic' in x if x else False)
            if not topics:
                break
            
            for t in topics:
                title_elem = t.find('h2', class_='title')
                if title_elem:
                    link = title_elem.find('a')
                    if link:
                        title = link.get('title', '') or link.get_text(strip=True)
                        date_elem = t.find('li', class_='date')
                        date_str = date_elem.get_text(strip=True)[:20] if date_elem else ''
                        
                        # Парсим дату
                        pub_date = None
                        try:
                            parts = date_str.replace(',', '').split()
                            if len(parts) >= 3:
                                day = int(parts[0])
                                month = month_map.get(parts[1], 1)
                                year = int(parts[2])
                                pub_date = datetime(year, month, day).date()
                        except:
                            pass
                        
                        # Фильтр по дате
                        if pub_date != target_date:
                            continue
                        
                        all_news.append({'date': date_str, 'title': title})
            
            # Если нашли достаточно - выходим
            if len(all_news) >= 3:
                break
                
        except Exception as e:
            print(f'Ошибка на странице {page}: {e}')
            break
    
    NEWS_CACHE_DATE[cache_key] = all_news
    return all_news


# Тест: новости за конкретную дату
print("=== Тест: SBER за 2026-04-28 ===")
news = get_company_news_by_date('sber', '2026-04-28')
print(f'Новостей: {len(news)}')
for n in news[:3]:
    print(f'  - {n["title"][:80]}')

print("\n=== Тест: OZON за 2025-11-10 ===")
news = get_company_news_by_date('ozon', '2025-11-10')
print(f'Новостей: {len(news)}')
for n in news[:3]:
    print(f'  - {n["title"][:80]}')

In [ ]:
import yfinance as yf

# Укажите тикер, например, Apple (AAPL)
ticker = yf.Ticker("OZON")

# Получение последних новостей
news = ticker.news

# Вывод заголовков новостей
for item in news:
    print(item['title'], item['link'])
